# Fase 1 — Reporte de reconciliación

**Proyecto:** CRM-Granos-MX  
**Fecha:** 2026-04-30

Este notebook es complementario al documento principal de la fase: [`docs/plan_ajustado.md`](../docs/plan_ajustado.md). El documento es la fuente de verdad textual; este notebook es para visualizaciones que ayudan a comunicar.

## Estructura

1. Resumen del plan ajustado (ver doc).
2. Diagrama de fases con dependencias y tiempos.
3. Distribución de costos estimados.
4. Decisiones pendientes (ver doc §3).

## 1. Plan ajustado en una imagen

```
Fase 0 ✅ Arqueología + scaffold                               (cerrada 2026-04-30)
Fase 1 ✅ Reporte de reconciliación                            (cerrada 2026-04-30)
Fase 1.bis  Cimientos: config, db, models, logging, compliance helper, CI   [2 días]
Fase 2      Diseño de chequeos de calidad                       [3 días]
Fase 3      Descarga real DENUE (104 batches, ~125K registros)  [5 días]   ← camino crítico
Fase 4      Enriquecimiento Google Places                       [4 días + ~$25 USD]
Fase 5      Cruce con datos públicos adicionales                [5 días núcleo + 3 opc]
Fase 6      Algoritmo de scoring + asignación                   [4 días + workshop]
Fase 7      Compliance LFPDPPP 2025                             [4 días + revisión legal]
Fase 8      API + Dashboard + deploy Render                     [7-12 días + ~$30 USD/mes]
                                                                ──────────────
                                                       Total ≈ 37-42 días-hombre
                                                       ≈ 2-3 meses calendario
```

**Camino crítico:** Fase 3 (DENUE descarga) bloquea todo lo posterior. Fase 1.bis es prerrequisito de Fase 2-8.

## 2. Decisiones pendientes (ver `docs/plan_ajustado.md` §3)

| # | Tema | Bloqueante | Recomendación Claude |
|---|---|---|---|
| D1 | Fase 1.bis aparte vs integrada en Fase 2 | 🟡 | Aparte |
| D2 | Estilo modelos ORM (Mapped[...] vs clásico) | 🟡 | Mapped[...] (SQLAlchemy 2.0) |
| D3 | Estrategia costos Google Places | 🔴 | D3.B Balanceado (~$25 USD) |
| D4 | Frontend dashboard (server-rendered vs SPA) | 🔴 | D4.A Server-rendered FastAPI+Jinja+HTMX |
| D5 | Cuándo tramitar Google Places key | 🟡 | D5.A Ahora (Fase 1.bis) |
| D6 | Aviso privacidad: quién redacta | 🟡 | D6.A Yo paso draft, tú lo revisas con asesor |
| D7 | Calibración scoring | 🔴 | D7.C Workshop + tracking automático |

## 3. Visualización (placeholder hasta Fase 3)

La celda de abajo dibuja un mapa coroplético de las 13 entidades priorizadas con el dimensionamiento Cuantificar de Fase 0. Requiere `geopandas`, `matplotlib`, `contextily` (en deps `[notebooks]` de pyproject.toml — instalar con `pip install -e .[notebooks]`).

Si esto no corre todavía es porque las deps de notebook no están instaladas. No es bloqueante para Fase 1.bis.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().parent
DIM = ROOT / "data" / "raw" / "dimensionamiento_inicial.json"

if not DIM.exists():
    print("data/raw/dimensionamiento_inicial.json no encontrado.")
    print("Si vienes de una clonada limpia, regenera el archivo con el snippet de Fase 0.")
else:
    dim = json.loads(DIM.read_text())
    print(f"Plan apunta a {dim['total_priorizados_5_scian']:,} establecimientos")
    print(f"en {len(dim['entidades_priorizadas'])} entidades, repartidos en 4 canales.")
    print("\nDistribución por canal (al cierre de Fase 0):")
    for canal, valores in dim['por_canal'].items():
        print(f"  {canal:<25} {valores['priorizados']:>8,} en priorizados")

## 4. Próximo paso

Cuando el operador apruebe las 7 decisiones de §3 del plan, arrancamos **Fase 1.bis — Cimientos transversales**:
1. `src/core/config.py` con Pydantic Settings
2. `src/core/db.py` con engine SQLAlchemy + sesión
3. `src/core/logging.py` con loguru
4. `src/core/models.py` con modelos ORM declarativos
5. `src/core/repos/` con repositorios por dominio
6. `src/compliance/lfpdppp.py` con helper `registrar_operacion()`
7. `tests/conftest.py` con fixture Postgres efímero
8. `.github/workflows/ci.yml` con pytest + ruff + alembic check
9. Instalación de las deps que faltan (~30 paquetes)

Estimación 2 días.